# 01 · 원자료 수집과 전처리 재현

상명풍력 발전량 + 새별오름 AWS 883 → 시간별 병합 데이터. 이 문서는 이미 수집한 원자료의 위치·무결성·변환을 재현합니다. 자동 다운로드나 수집되지 않은 메타데이터를 만들어내지 않습니다. 모든 계산은 `src/data/*.py`, 호출 안내는 `src/workflows/research.py`에 있습니다.

**실행:** `Python (wind_power)` 커널에서 위→아래 Run All. 노트북은 각각 독립 실행할 수 있습니다. 실제 재학습은 03/04의 명시적 옵션에서만 시작합니다.

In [1]:
from pathlib import Path
import os, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/base_config.yaml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
runtime = ROOT / ".experiment_archive/python_runtime"
if runtime.exists(): sys.path.insert(0, str(runtime))
from IPython.display import display
from src.workflows import research as wf
%matplotlib inline

In [2]:
wf.environment()

,component,version
0,python,3.11.16
1,numpy,2.2.6
2,scipy,1.15.3
3,pandas,3.0.6
4,torch,2.6.0+cu124
5,lightgbm,4.7.0
6,prophet,1.4.0
7,device,NVIDIA GeForce RTX 4080 SUPER


## 1. 데이터 출처와 로컬 원본

- 발전량: [한국중부발전 상명풍력 발전 실적](https://www.data.go.kr/data/15119454/fileData.do). `data/raw/generation/`의 wide CSV(기준일, 1시~24시).
- 기상: [기상청 AWS](https://data.kma.go.kr/), 새별오름 883. `data/raw/weather/saebyeol_883-hour/`의 2023·2024·2025 시간 CSV.
- 분 자료의 시간자료 재구성 감사는 `src/experiments/audit_aws_reconstruction.py` 및 `reports/aws_reconstruction_validation.json`에 별도로 보존합니다. 2026 후보 기상은 본 실험에 병합하지 않습니다.

아래 SHA-256은 **현재 파일**의 식별자입니다. 다운로드 당시 제공기관 설명·정비 이력·전달 지연을 증명하는 값은 아닙니다.

In [3]:
wf.source_inventory()

,path,bytes,sha256
0,data/raw/generation/한국중부발전(주)_(AI 친화 데이터)풍력 발전...,214856,f8da66c0a92f0454c9b1e4036edccca9546388096562de...
1,data/raw/weather/saebyeol_883-hour/SURFACE_AWS...,499517,fded8d75c0a7338deae732fc6f7842ea9664f317830ffc...
2,data/raw/weather/saebyeol_883-hour/SURFACE_AWS...,498195,1c2b4769b343aedb247df218f49ecd9cb7ac7d3830af02...
3,data/raw/weather/saebyeol_883-hour/SURFACE_AWS...,497540,068100dfc08943d18bf7e45a1a2803b36b748edae45b81...


## 2. 변환 규칙과 확인되지 않은 가정

발전량 단위와 시간 집계 경계는 공식적으로 확정되지 않았습니다. 원자료의 스케일 불연속을 바탕으로 코드가 사용하는 가정을 아래에 명시합니다. 21 MWh 상한은 21 MW × 가정한 1시간 구간입니다. 기관 설명이 확인되면 가정과 데이터 버전을 재검토해야 합니다.

In [4]:
wf.protocol_table()

,item,definition
0,target,generation_mwh; exact observed 0 is the zero l...
1,unit assumption,before 2025-02-01: raw/1e6; afterwards: raw/1e...
2,time assumption,"1-hour energy, interval-end label; 24:00 = nex..."
3,capacity,21 MWh per assumed one-hour interval
4,input,"24 hours: generation + wind speed, wind sin/co..."
5,missing values,"hourly grid, weather past-only ffill <=3h; inv..."
6,scaling,generation /21; weather mean/std fitted on tra...
7,selection,2024 H2 validation CRPS; fixed-rule models hav...
8,evaluation,2025 development benchmark; not an independent...
9,2026,not forecast or scored by these notebooks


## 3. 원본 → generation_hourly → weather_hourly → merged_dataset

`load_generation.build_and_save_generation_interim()` → `process_saebyeol_weather.build_and_save_master_dataset()`를 호출합니다. 24시는 다음 날 00시로 변환하고, 영발전 0은 그대로 보존합니다. 기상은 정규 시간격자에서 과거값만 최대 3시간 채우며 원래 결측·보간 여부·관측 나이를 기록합니다.

기존 데이터는 덮어쓰지 않습니다. `.experiment_archive/notebook_reproduction/`에 다시 만들고 기존 45개 열 전체와 정확히 일치하는지 검사합니다. 이 단계는 파일 전체를 변환하지만 2026년 예측·성능 평가는 하지 않습니다.

In [5]:
wf.rebuild_preprocessing()

[*] Processing Saebyeol-Oreum AWS 883 weather...
[*] Found 3 Saebyeol 883 weather CSV files:
    - data/raw/weather/saebyeol_883-hour\SURFACE_AWS_883_HR_2023_2023_2024.csv
    - data/raw/weather/saebyeol_883-hour\SURFACE_AWS_883_HR_2024_2024_2025.csv
    - data/raw/weather/saebyeol_883-hour\SURFACE_AWS_883_HR_2025_2025_2026.csv
[+] Saved clean hourly weather (26,304 rows) to: D:\InternetDownload\Lab\Wind Power\.experiment_archive\notebook_reproduction\weather_hourly.parquet
[*] Loading generation data from: D:\InternetDownload\Lab\Wind Power\.experiment_archive\notebook_reproduction\generation_hourly.parquet
[+] Saved master merged dataset (28,440 rows) to: D:\InternetDownload\Lab\Wind Power\.experiment_archive\notebook_reproduction\merged_dataset.parquet


,status,rows,columns,original_sha256,rebuilt_sha256,output_directory
0,exact_values_match,28440,45,01861a5a18cac3731f4238d9307d03e2bb29d9c3aa11e8...,01861a5a18cac3731f4238d9307d03e2bb29d9c3aa11e8...,D:\InternetDownload\Lab\Wind Power\.experiment...


## 4. 연구 구간의 결측과 분할

학습: 2023~2024 상반기, 검증: 2024 하반기, 개발 평가: 2025. 아래 요약과 다음 노트북 EDA는 2025년 말까지만 사용합니다. 2025년은 여러 차례 모델 개발에 참조한 구간이므로 독립 테스트라고 부르지 않습니다.

In [6]:
wf.split_quality()

,split,start,end,rows,generation_missing,zero_count,zero_pct,incomplete_weather_rows
0,train,2023-01-01 01:00:00,2024-06-30 23:00:00,13127,0,2391,18.214367,10
1,validation,2024-07-01 00:00:00,2024-12-31 23:00:00,4416,0,982,22.237319,51
2,test,2025-01-01 00:00:00,2025-12-31 23:00:00,8760,0,1426,16.278539,0


In [7]:
wf.weather_quality()

,variable,original_missing,forward_filled,remaining_missing
0,aws_wind_speed,37,18,19
1,aws_wind_direction,37,18,19
2,aws_temperature,25,12,13
3,aws_humidity,77,22,55
4,aws_local_pressure,24,11,13


**다음:** [02 · 실제 모델 입력 EDA](02_model_dataset_eda.ipynb). 상세 구현: `src/data/load_generation.py`, `src/data/process_saebyeol_weather.py`, `src/data/forecast_protocol.py`.